# 00. 데이터 점검 노트북 (Data Check)

> **제5회 ETRI 휴먼이해 인공지능 논문경진대회**  
> 데이터를 처음 받았을 때 실행하여 구조를 파악하세요.  
> 실제 데이터가 없으면 친절한 안내 메시지가 출력됩니다.

---

## 실행 순서
1. `data/raw/` 또는 `/data/`에 대회 데이터 배치
2. 셀 순서대로 실행 (`Shift+Enter`)
3. 결과를 보고 `configs/base.yaml` 업데이트
4. 하단 체크리스트 확인 후 baseline 학습 진행

## 0. 환경 설정

In [ ]:
# -*- coding: utf-8 -*-
import sys
import os

# 프로젝트 루트를 sys.path에 추가 (notebooks/ 에서 src/ 임포트)
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"Project root: {project_root}")
print(f"Python: {sys.version}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# 시각화 설정
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_style('whitegrid')

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.4f}'.format)

print("라이브러리 로드 완료")

## 1. Config 로드

In [ ]:
from src.config import Config

cfg = Config("configs/base.yaml")
print(cfg)
print()
print("경로 확인:")
for k, v in cfg.paths.items():
    exists = "✓ 존재" if v.exists() else "✗ 없음"
    print(f"  {k:25s}: {v}  [{exists}]")

## 2. 데이터 로드

> 데이터 파일이 없으면 어떤 파일을 어디에 넣어야 하는지 안내가 출력됩니다.

In [ ]:
from src.load_data import load_train, load_test, load_sample_submission

# 데이터 로드 (파일 없으면 FileNotFoundError + 친절한 안내 출력)
try:
    train = load_train(cfg)
    test  = load_test(cfg)
    sub   = load_sample_submission(cfg)
    print("\n✓ 데이터 로드 성공!")
    DATA_LOADED = True
except FileNotFoundError as e:
    print(e)
    print()
    print("📌 데이터를 넣은 후 이 셀부터 다시 실행하세요.")
    DATA_LOADED = False

## 3. 데이터 Shape 확인

In [ ]:
if DATA_LOADED:
    print(f"{'='*50}")
    print(f"  Train  : {train.shape[0]:,} rows × {train.shape[1]:,} cols")
    print(f"  Test   : {test.shape[0]:,} rows × {test.shape[1]:,} cols")
    print(f"  Sample : {sub.shape[0]:,} rows × {sub.shape[1]:,} cols")
    print(f"{'='*50}")
else:
    print("데이터를 먼저 로드하세요. (셀 2 참고)")

## 4. 컬럼 목록 확인

In [ ]:
if DATA_LOADED:
    print(f"[Train 컬럼 - {len(train.columns)}개]")
    for i, col in enumerate(train.columns):
        dtype = train[col].dtype
        print(f"  {i+1:3d}. {col:35s} {str(dtype):10s}")
else:
    print("데이터를 먼저 로드하세요.")

In [ ]:
if DATA_LOADED:
    print(f"[Test 컬럼 - {len(test.columns)}개]")
    for i, col in enumerate(test.columns):
        dtype = test[col].dtype
        print(f"  {i+1:3d}. {col:35s} {str(dtype):10s}")

## 5. Train/Test 공통 컬럼 vs 차이 컬럼

In [ ]:
if DATA_LOADED:
    train_cols = set(train.columns)
    test_cols  = set(test.columns)
    
    common = train_cols & test_cols
    only_train = train_cols - test_cols
    only_test  = test_cols  - train_cols
    
    print(f"공통 컬럼 수   : {len(common)}개")
    print(f"Train에만 있음 : {len(only_train)}개  → {sorted(only_train)}")
    print(f"Test에만 있음  : {len(only_test)}개   → {sorted(only_test)}")
    
    if only_train:
        print()
        print("💡 Train에만 있는 컬럼은 target일 가능성이 높습니다.")
        print(f"   후보: {sorted(only_train)}")

## 6. Target 컬럼 추론

In [ ]:
if DATA_LOADED:
    from src.load_data import infer_id_col, infer_target_cols
    
    # sample_submission 컬럼 확인
    print(f"[Sample Submission 컬럼]")
    print(f"  {list(sub.columns)}")
    print()
    
    # id_col 추론
    if cfg.id_col:
        id_col = cfg.id_col
        print(f"id_col (config에서 지정): '{id_col}'")
    else:
        id_col = infer_id_col(sub)
    
    # target_cols 추론
    if cfg.target_cols:
        target_cols = cfg.target_cols
        print(f"target_cols (config에서 지정): {target_cols}")
    else:
        target_cols = infer_target_cols(sub, id_col)
    
    print()
    print(f"✓ id_col      : '{id_col}'")
    print(f"✓ target_cols : {target_cols}")
    print()
    print("💡 이 값이 맞으면 configs/base.yaml에 명시적으로 설정하세요:")
    print(f"   data.id_col: \"{id_col}\"")
    print(f"   data.target_cols: {target_cols}")

## 7. ID 컬럼 추론 및 검증

In [ ]:
if DATA_LOADED:
    print(f"[ID 컬럼 '{id_col}' 분석]")
    print(f"  Train 고유값 수: {train[id_col].nunique():,}")
    print(f"  Test  고유값 수: {test[id_col].nunique():,}")
    
    train_ids = set(train[id_col].unique())
    test_ids  = set(test[id_col].unique())
    overlap   = train_ids & test_ids
    only_train_ids = train_ids - test_ids
    only_test_ids  = test_ids  - train_ids
    
    print(f"  Train/Test 공통 ID: {len(overlap):,}개")
    print(f"  Train에만 있는 ID: {len(only_train_ids):,}개")
    print(f"  Test에만 있는 ID:  {len(only_test_ids):,}개")
    
    if len(overlap) == 0:
        print()
        print("⚠ Train과 Test의 ID가 겹치지 않습니다.")
        print("  → 시계열 예측 또는 시간 기반 분할 가능성 있음")

## 8. 결측치 비율 확인

In [ ]:
if DATA_LOADED:
    from src.load_data import print_data_summary
    
    print_data_summary(train, "Train")
    print_data_summary(test,  "Test")

In [ ]:
if DATA_LOADED:
    # 결측치 비율 시각화 (상위 20개 컬럼)
    missing_ratio = train.isna().mean().sort_values(ascending=False)
    missing_nonzero = missing_ratio[missing_ratio > 0].head(20)
    
    if len(missing_nonzero) > 0:
        fig, ax = plt.subplots(figsize=(10, max(4, len(missing_nonzero) * 0.4)))
        missing_nonzero.sort_values().plot(kind='barh', ax=ax, color='salmon')
        ax.set_xlabel('결측치 비율')
        ax.set_title('Train 결측치 비율 (상위 20개)')
        ax.axvline(x=0.5, color='red', linestyle='--', alpha=0.5, label='50%')
        ax.legend()
        plt.tight_layout()
        plt.show()
    else:
        print("Train 데이터에 결측치가 없습니다. ✓")

## 9. 기초 통계량 확인

In [ ]:
if DATA_LOADED:
    print("[Train 기초 통계량 - 수치형]")
    display(train.describe())

In [ ]:
if DATA_LOADED:
    # Target 컬럼 분포
    if target_cols:
        print("[Target 컬럼 분포]")
        n_targets = len(target_cols)
        fig, axes = plt.subplots(1, n_targets, figsize=(6 * n_targets, 4))
        if n_targets == 1:
            axes = [axes]
        
        for ax, col in zip(axes, target_cols):
            if col in train.columns:
                train[col].hist(bins=30, ax=ax, color='steelblue', edgecolor='white')
                ax.set_title(f'Target: {col}')
                ax.set_xlabel('Value')
                ax.set_ylabel('Count')
                # 기초 통계
                stats = train[col].describe()
                ax.text(0.02, 0.98, 
                        f"mean={stats['mean']:.3f}\nstd={stats['std']:.3f}\nmin={stats['min']:.3f}\nmax={stats['max']:.3f}",
                        transform=ax.transAxes, verticalalignment='top',
                        fontsize=9, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        
        plt.tight_layout()
        plt.show()

## 10. 날짜형 컬럼 후보 탐지

In [ ]:
if DATA_LOADED:
    keywords = ["date", "time", "dt", "timestamp", "날짜", "일자"]
    date_candidates = []
    
    for col in train.columns:
        # 1. 컬럼명 패턴
        if any(kw in col.lower() for kw in keywords):
            date_candidates.append((col, "컬럼명 패턴"))
            continue
        # 2. object 타입 컬럼에서 날짜 파싱 시도
        if train[col].dtype == object:
            sample = train[col].dropna().head(5)
            try:
                pd.to_datetime(sample)
                date_candidates.append((col, "파싱 가능"))
            except:
                pass
    
    if date_candidates:
        print("[날짜형 컬럼 후보]")
        for col, reason in date_candidates:
            sample_val = train[col].dropna().iloc[0] if len(train[col].dropna()) > 0 else "N/A"
            print(f"  - {col:35s} ({reason}) 예: {sample_val}")
        print()
        print("💡 configs/base.yaml에서 명시적으로 지정할 수 있습니다:")
        cols_str = str([c for c, _ in date_candidates])
        print(f"   features.date_cols: {cols_str}")
    else:
        print("날짜형 컬럼 후보를 찾지 못했습니다.")
        print("use_date_features: false로 유지하세요.")

## 11. User_id / Subject_id 컬럼 후보 탐지

In [ ]:
if DATA_LOADED:
    user_keywords = ["user", "subject", "person", "participant", "uid", "pid"]
    user_candidates = []
    
    for col in train.columns:
        if any(kw in col.lower() for kw in user_keywords):
            n_unique = train[col].nunique()
            user_candidates.append((col, n_unique))
    
    if user_candidates:
        print("[개인 식별 컬럼 후보]")
        for col, n_unique in user_candidates:
            rows_per_user = len(train) // n_unique if n_unique > 0 else 0
            print(f"  - {col:35s} 고유값={n_unique:4d}개  (평균 {rows_per_user}행/사람)")
        print()
        print("💡 개인별 기준선 피처와 GroupKFold를 위해 configs/base.yaml에 설정:")
        best_col = user_candidates[0][0]
        print(f"   features.group_col: \"{best_col}\"")
        print(f"   training.use_group_kfold: true")
        print(f"   training.group_col: \"{best_col}\"")
    else:
        print("개인 식별 컬럼 후보를 찾지 못했습니다.")
        print("GroupKFold 없이 일반 KFold를 사용하세요.")

## 12. Target 컬럼 기초 분석

In [ ]:
if DATA_LOADED and target_cols:
    print("[Target 컬럼 기초 통계]")
    display(train[target_cols].describe())
    print()
    
    # Target 상관관계 (multi-target일 경우)
    if len(target_cols) > 1:
        print("[Target 간 상관관계]")
        corr = train[target_cols].corr()
        display(corr.round(3))
        
        fig, ax = plt.subplots(figsize=(max(6, len(target_cols)), max(5, len(target_cols))))
        sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm', ax=ax, 
                    center=0, vmin=-1, vmax=1)
        ax.set_title('Target 간 상관관계')
        plt.tight_layout()
        plt.show()

## 13. 피처 간 상관관계 (Target과의 상관)

In [ ]:
if DATA_LOADED and target_cols:
    # 수치형 피처와 target 간 상관관계
    num_cols = train.select_dtypes(include=['int64', 'float64']).columns.tolist()
    feat_cols_num = [c for c in num_cols if c not in target_cols and c != id_col]
    
    if feat_cols_num:
        for target in target_cols:
            if target in train.columns:
                corr_with_target = train[feat_cols_num + [target]].corr()[target]\
                    .drop(target).abs().sort_values(ascending=False)
                
                print(f"['{target}'와 상관관계 높은 피처 Top 20]")
                display(corr_with_target.head(20).to_frame('|correlation|'))
                print()

## 14. Baseline 학습 전 체크리스트

In [ ]:
if DATA_LOADED:
    print("=" * 60)
    print("  Baseline 학습 전 체크리스트")
    print("=" * 60)
    
    checks = []
    
    # 1. 데이터 로드
    checks.append(("✓", f"Train 로드: {train.shape[0]:,} rows × {train.shape[1]:,} cols"))
    checks.append(("✓", f"Test  로드: {test.shape[0]:,} rows × {test.shape[1]:,} cols"))
    
    # 2. id_col 확인
    if id_col in train.columns:
        checks.append(("✓", f"id_col = '{id_col}' (Train에 존재)"))
    else:
        checks.append(("✗", f"id_col = '{id_col}' → Train에 없음! configs/base.yaml 확인 필요"))
    
    # 3. target_cols 확인
    missing_targets = [c for c in target_cols if c not in train.columns]
    if not missing_targets:
        checks.append(("✓", f"target_cols = {target_cols} (모두 Train에 존재)"))
    else:
        checks.append(("✗", f"target_cols 누락: {missing_targets}"))
    
    # 4. 결측치
    total_missing = train.isna().sum().sum()
    if total_missing == 0:
        checks.append(("✓", "Train 결측치 없음"))
    else:
        missing_ratio_total = total_missing / (train.shape[0] * train.shape[1])
        checks.append(("△", f"Train 결측치 {total_missing:,}개 ({missing_ratio_total*100:.1f}%) → preprocess에서 자동 처리"))
    
    # 5. 날짜 피처
    if date_candidates:
        checks.append(("💡", f"날짜형 컬럼 후보: {[c for c,_ in date_candidates]} → use_date_features: true 권장"))
    else:
        checks.append(("△", "날짜형 컬럼 없음 → use_date_features: false 유지"))
    
    # 6. 개인 식별 컬럼
    if user_candidates:
        checks.append(("💡", f"개인 식별 컬럼 후보: {[c for c,_ in user_candidates]} → GroupKFold 권장"))
    else:
        checks.append(("△", "개인 식별 컬럼 없음 → KFold 사용"))
    
    for symbol, msg in checks:
        print(f"  {symbol} {msg}")
    
    print("=" * 60)
    print()
    print("모든 ✓ 확인 후 baseline 학습 실행:")
    print("  python src/train_lgbm.py --config configs/base.yaml")
    print("  또는")
    print("  powershell -ExecutionPolicy Bypass -File scripts/run_baseline.ps1")

else:
    print("=" * 60)
    print("  ⚠ 데이터를 먼저 배치하세요")
    print("=" * 60)
    print()
    print("다음 위치에 대회 데이터를 배치하세요:")
    print(f"  data/raw/train.csv")
    print(f"  data/raw/test.csv")
    print(f"  data/raw/sample_submission.csv")
    print()
    print("또는 configs/base.yaml에서 경로를 변경하세요:")
    print("  paths.data_dir: \"data/raw\"  # 기본")
    print("  paths.data_dir: \"/data\"      # 대회 서버")

---

## 다음 단계

1. **데이터 확인 완료** → `configs/base.yaml`에 `id_col`, `target_cols` 명시적 설정
2. **날짜 컬럼 확인** → `features.date_cols` 설정
3. **개인 식별 컬럼 확인** → `training.use_group_kfold: true`, `group_col` 설정
4. **Baseline 학습**: `python src/train_lgbm.py --config configs/base.yaml`
5. **결과 기록**: `docs/experiment_log.md` Exp001 업데이트
6. **EDA 심화**: `notebooks/eda.ipynb`에서 추가 분석